# Introduction to Vector Databases (Installing and Running ChromaDB)

**Level:** Scratch → Intermediate
**Runs on:** Google Colab (Free CPU)

### What you will learn
1. What a Vector Database is and why we need one
2. Installing ChromaDB (free, open-source, runs locally)
3. Creating a Collection (like a table, but for vectors)
4. Adding documents (ChromaDB auto-generates embeddings)
5. Querying by meaning (semantic search)
6. Using your own Hugging Face embeddings with ChromaDB (intermediate)
7. Updating and deleting records

> No API key, no cloud account, no cost. Everything runs in-memory inside this notebook.

## 1. Why a Vector Database?

A normal database (SQL) searches by **exact match** — e.g., `WHERE name = 'dog'`.

A **Vector Database** stores embeddings (numeric meaning-vectors) and searches by **similarity** — e.g., "find text that *means* something like 'dog'", even if the word "dog" never appears.

| Normal Database | Vector Database |
|---|---|
| Matches exact text/values | Matches by meaning |
| Uses SQL queries | Uses similarity search |
| Good for structured data | Good for text, images, audio embeddings |

**ChromaDB** is one of the simplest free, open-source vector databases — great for learning.

### 2. Install ChromaDB
Run this once. `-q` keeps the output clean.

In [1]:
# !pip install -q chromadb

### 3. Create a ChromaDB Client

We use an **in-memory / ephemeral client** — perfect for learning in Colab.
(Data will reset when the runtime restarts. For permanent storage, `chromadb.PersistentClient(path="...")` is used instead — shown later.)

In [2]:
import chromadb

client = chromadb.Client()

print("ChromaDB client created successfully!")

ChromaDB client created successfully!


### 4. Create a Collection

A **Collection** in ChromaDB is like a table — it stores your documents, their embeddings, and metadata.

In [3]:
collection = client.create_collection(name="my_first_collection")

print("Collection created:", collection.name)

Collection created: my_first_collection


### 5. Add Documents

We simply give ChromaDB plain text. By default, it **automatically converts text into embeddings** for us using a built-in free model — no extra setup needed.

In [4]:
collection.add(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers.",
        "Paris is the capital city of France."
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5", "doc6"]
)

print("Documents added! Total documents:", collection.count())

Documents added! Total documents: 6


**Discussion Point:** Every document got an `id` (like a primary key). Behind the scenes, ChromaDB converted each sentence into an embedding vector and stored it — just like we did manually in the earlier embeddings notebook.

### 6. Query by Meaning (Semantic Search)

Now we ask a question in natural language. ChromaDB converts our query into an embedding too, then finds the closest matching documents.

In [5]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 0.9317  |  Deep learning uses neural networks with many layers.
Distance: 0.9516  |  Python is a popular programming language for AI.


**Discussion Point:** Lower `distance` = more similar (closer meaning). Notice the AI/ML related sentences come back first, even though the query never used the exact same words.

In [6]:
# Try another query
results = collection.query(
    query_texts=["What is the capital of France?"],
    n_results=2
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 0.3515  |  Paris is the capital city of France.
Distance: 1.7959  |  Python is a popular programming language for AI.


### 7. Adding Metadata (Intermediate)

Real-world documents usually have extra info (category, author, date). ChromaDB lets us store this alongside each document and filter on it.

In [7]:
collection.add(
    documents=[
        "The new smartphone has an amazing camera.",
        "Stock prices rose sharply after the earnings report."
    ],
    metadatas=[
        {"category": "technology"},
        {"category": "finance"}
    ],
    ids=["doc7", "doc8"]
)

# Query only within the "technology" category
results = collection.query(
    query_texts=["Tell me about new gadgets"],
    n_results=2,
    where={"category": "technology"}
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 1.1460  |  The new smartphone has an amazing camera.


### 8. Using Your Own Hugging Face Embeddings (Intermediate)

By default ChromaDB uses its own built-in embedding model. But we can plug in **any** Hugging Face model, such as `all-MiniLM-L6-v2` from the earlier notebook, for full control.

In [8]:
# !pip install -q sentence-transformers

In [9]:
from chromadb.utils import embedding_functions

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create a new collection that uses our chosen Hugging Face model
custom_collection = client.create_collection(
    name="custom_embedding_collection",
    embedding_function=hf_embedder
)

custom_collection.add(
    documents=[
        "I love dogs",
        "I like puppies",
        "The stock market crashed today"
    ],
    ids=["a1", "a2", "a3"]
)

results = custom_collection.query(
    query_texts=["puppy lovers"],
    n_results=2
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Distance: 0.2513  |  I like puppies
Distance: 0.3943  |  I love dogs


### 9. Update and Delete Records

In [10]:
# Update a document's text
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."]
)
print("Updated doc4")

# Delete a document
collection.delete(ids=["doc2"])
print("Deleted doc2")

print("Remaining documents:", collection.count())

Updated doc4
Deleted doc2
Remaining documents: 7


### 10. Persistent Storage (So Data Survives a Restart)

The examples above used an in-memory client, which resets on restart. For real projects, use `PersistentClient` to save data to disk.

In [11]:
persistent_client = chromadb.PersistentClient(path="/content/chroma_db")

persistent_collection = persistent_client.get_or_create_collection(name="persistent_demo")

persistent_collection.add(
    documents=["This data will be saved to disk."],
    ids=["p1"]
)

print("Saved to disk at /content/chroma_db")
print("Count:", persistent_collection.count())

Saved to disk at /content/chroma_db
Count: 1


### 11. Key Takeaways (Recap for Students)

| Concept | Meaning |
|---|---|
| Vector Database | Stores embeddings, searches by meaning/similarity |
| ChromaDB | Free, open-source, runs locally — no cloud needed |
| Collection | Like a table — holds documents + embeddings + metadata |
| `add()` | Insert documents (embeddings created automatically) |
| `query()` | Search by meaning, returns closest matches |
| `where={}` | Filter results using metadata |
| `PersistentClient` | Saves the database to disk instead of memory |

### 12. Practice Exercises
1. Add 5 of your own sentences to a new collection and query them with a related question.
2. Add a `"category"` metadata field to every document, then filter a query by two different categories.
3. Swap the embedding function to `'all-mpnet-base-v2'` and compare the distance scores to `all-MiniLM-L6-v2`.

In [12]:
sentences = [
    "Python is widely used for artificial intelligence.",
    "Machine learning allows computers to learn from data.",
    "Vector databases store embeddings for similarity search.",
    "ChromaDB is a popular vector database for AI applications.",
    "RAG systems combine retrieval with generative AI."
]

collection = client.create_collection(name="practice_sentences")

collection.add(
    documents=sentences,
    ids=["p1", "p2", "p3", "p4", "p5"]
)

results = collection.query(
    query_texts=["How are vector databases used in AI?"],
    n_results=2
)

print(results)

{'ids': [['p4', 'p3']], 'embeddings': None, 'documents': [['ChromaDB is a popular vector database for AI applications.', 'Vector databases store embeddings for similarity search.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.7034308910369873, 0.9333705306053162]]}


In [13]:
sentences = [
    "Python is widely used for artificial intelligence.",
    "Machine learning allows computers to learn from data.",
    "Vector databases store embeddings for similarity search.",
    "ChromaDB is a popular vector database for AI applications.",
    "RAG systems combine retrieval with generative AI."
]

categories = [
    "AI",
    "AI",
    "Database",
    "Database",
    "RAG"
]

In [14]:
collection = client.create_collection(name="practice_categories")

collection.add(
    documents=sentences,
    ids=["p1", "p2", "p3", "p4", "p5"],
    metadatas=[
        {"category": "AI"},
        {"category": "AI"},
        {"category": "Database"},
        {"category": "Database"},
        {"category": "RAG"}
    ]
)

In [15]:
results = collection.query(
    query_texts=["How are vector databases used in AI?"],
    n_results=2,
    where={"category": "Database"}
)

print(results)

{'ids': [['p4', 'p3']], 'embeddings': None, 'documents': [['ChromaDB is a popular vector database for AI applications.', 'Vector databases store embeddings for similarity search.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'category': 'Database'}, {'category': 'Database'}]], 'distances': [[0.7034308910369873, 0.9333705306053162]]}


In [16]:
results_ai = collection.query(
    query_texts=["How is AI related to machine learning?"],
    n_results=2,
    where={"category": "AI"}
)

print(results_ai)

{'ids': [['p2', 'p1']], 'embeddings': None, 'documents': [['Machine learning allows computers to learn from data.', 'Python is widely used for artificial intelligence.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'category': 'AI'}, {'category': 'AI'}]], 'distances': [[0.5939637422561646, 1.0715932846069336]]}


In [17]:
from chromadb.utils import embedding_functions

mpnet_embedding = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2"
)

print("all-mpnet-base-v2 embedding function created")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Bhanu\.gemini\antigravity\scratch\gen-ai-tasks\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Bhanu\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

C:\Users\Bhanu\.gemini\antigravity\scratch\gen-ai-tasks\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Bhanu\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

In [18]:
mpnet_collection = client.create_collection(
    name="practice_mpnet",
    embedding_function=mpnet_embedding
)

mpnet_collection.add(
    documents=sentences,
    ids=["p1", "p2", "p3", "p4", "p5"]
)

print("Documents added using all-mpnet-base-v2")

Documents added using all-mpnet-base-v2


In [19]:
mpnet_results = mpnet_collection.query(
    query_texts=["How are vector databases used in AI?"],
    n_results=2
)

print(mpnet_results)

{'ids': [['p4', 'p3']], 'embeddings': None, 'documents': [['ChromaDB is a popular vector database for AI applications.', 'Vector databases store embeddings for similarity search.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.4003336429595947, 0.45143675804138184]]}


In [20]:
print("MiniLM-L6-v2 distances:")
print(results["distances"])

print("\nMPNet distances:")
print(mpnet_results["distances"])

MiniLM-L6-v2 distances:
[[0.7034308910369873, 0.9333705306053162]]

MPNet distances:
[[0.4003336429595947, 0.45143675804138184]]


In [21]:
print("=== Embedding Model Comparison ===\n")

print("all-MiniLM-L6-v2")
print("Documents:", results["documents"][0])
print("Distances:", results["distances"][0])

print("\nall-mpnet-base-v2")
print("Documents:", mpnet_results["documents"][0])
print("Distances:", mpnet_results["distances"][0])

=== Embedding Model Comparison ===

all-MiniLM-L6-v2
Documents: ['ChromaDB is a popular vector database for AI applications.', 'Vector databases store embeddings for similarity search.']
Distances: [0.7034308910369873, 0.9333705306053162]

all-mpnet-base-v2
Documents: ['ChromaDB is a popular vector database for AI applications.', 'Vector databases store embeddings for similarity search.']
Distances: [0.4003336429595947, 0.45143675804138184]
